# Matched top-5 retrieval ablations

Enable GPU and internet. Attach the existing `colab_benchmark_inputs.zip` Dataset, then Run All.
The verified 3,196-question baseline rankings are embedded; no result ZIP or API key is needed.
Runs BM25, dense, and hybrid without BGE. No Qwen, BGE loading, refinement, generation or re-embedding.
A single GPU is sufficient for the query embedding model. CPU BM25 searches are shared across methods.
README results stay unchanged. Exports include raw checkpoint records and a matched comparison table.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, io, json, os, shutil, subprocess, sys, zipfile
REPO = Path('/kaggle/working/vietnamese-legal-chatbot')
REPO.mkdir(parents=True, exist_ok=True)
source = base64.b64decode('...')
assert hashlib.sha256(source).hexdigest() == '25abac760f6228129aa9acdb89d3efb3ae2f7fe23cc09e27cbc68003627fb3d7'
with zipfile.ZipFile(io.BytesIO(source)) as archive:
    archive.extractall(REPO)
os.chdir(REPO)
baseline_bytes = base64.b64decode('...')
assert hashlib.sha256(baseline_bytes).hexdigest() == 'd5d590a17cf25d37bc8c9edbe887f858aff1072894418a6e18a7ca2611dea50d'
baseline_data = json.loads(gzip.decompress(baseline_bytes))
BASELINE = REPO / 'results/ablation-baseline'
BASELINE.mkdir(parents=True, exist_ok=True)
(BASELINE / 'manifest.json').write_text(json.dumps(baseline_data['manifest']), encoding='utf-8')
(BASELINE / 'questions.jsonl').write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in baseline_data['rows']), encoding='utf-8')
(BASELINE / 'origin.json').write_text(json.dumps({'original_questions_sha256': baseline_data['original_questions_sha256']}), encoding='utf-8')
print('Current source and verified retrieval baseline restored.')

## Dependencies and dataset

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, 'download_dataset.py'], check=True)

## Restore existing indexes

In [ ]:
"""Accept Kaggle's ZIP or automatically unpacked Dataset layout."""
import hashlib
from pathlib import Path, PurePosixPath
import shutil
from zipfile import ZipFile, ZIP_STORED


def input_digest(path):
    with Path(path).open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()


def restore_kaggle_inputs(input_root, repo, expected):
    input_root, repo = Path(input_root), Path(repo)
    bundle = next(input_root.rglob('colab_benchmark_inputs.zip'), None)
    destination = repo / 'colab_vector_index.zip'
    if bundle:
        if input_digest(bundle) != expected['bundle_sha256']:
            raise ValueError('Index bundle checksum differs.')
        with ZipFile(bundle) as archive:
            if set(archive.namelist()) != {'bm25_index.pkl', 'colab_vector_index.zip'}:
                raise ValueError('Unexpected bundle contents.')
            (repo / 'index').mkdir(parents=True, exist_ok=True)
            with archive.open('bm25_index.pkl') as source, (repo / 'index/bm25_index.pkl').open('wb') as target:
                shutil.copyfileobj(source, target)
            archive.extract('colab_vector_index.zip', repo)
        print('Verified original ZIP input.')
        return destination

    bm25 = next((p for p in input_root.rglob('bm25_index.pkl') if input_digest(p) == expected['bm25_sha256']), None)
    if bm25 is None:
        raise ValueError('Matching BM25 index not found in the attached Kaggle Dataset.')
    vector_files = expected['vector_files']
    for name in vector_files:
        path = PurePosixPath(name)
        if path.is_absolute() or '..' in path.parts or '\\' in name or ':' in name:
            raise ValueError('Unsafe expected vector path.')
    manifests = [p for p in input_root.rglob('build_manifest.json')
                 if input_digest(p) == vector_files['build_manifest.json']]
    vector_root = None
    for manifest in manifests:
        root = manifest.parent
        if all((root / name).is_file() and input_digest(root / name) == checksum
               for name, checksum in vector_files.items()):
            vector_root = root
            break
    if vector_root is None:
        raise ValueError('Complete matching vector files not found; the extracted index is incomplete or changed.')
    # Verify all input hashes before changing any destination files.
    (repo / 'index').mkdir(parents=True, exist_ok=True)
    shutil.copyfile(bm25, repo / 'index/bm25_index.pkl')
    with ZipFile(destination, 'w', compression=ZIP_STORED) as archive:
        for name in vector_files:
            archive.write(vector_root / name, name)
    print('Verified automatically extracted Kaggle input and rebuilt the portable archive.')
    return destination


import os, subprocess, sys
EXPECTED_INPUTS = {'bundle_sha256': '9509bbe6b93c0fca3e41458f87c7b7b422ac803b87ff2c9babb2a918caae5d0e', 'bm25_sha256': 'b6dfc625509c717e0a8d213da26e61854945688b2299a133bd4a677e3e50f523', 'vector_files': {'build_manifest.json': '97b1e971626e6919213401a1228c3ae081fce614839afa8367cca48ae061398f', 'qdrant/meta.json': '77feeada277abff14b1a52c634379e3958f8e71ebb7db01839160dc359049239', 'qdrant/collection/bkai_biencoder_vietnamese_legal_corpus/storage.sqlite': 'dd6f58a96046140edee0cf9359086508de25f4fb3c1c26b9ff67998739c59f01'}}
archive_path = restore_kaggle_inputs('/kaggle/input', REPO, EXPECTED_INPUTS)
subprocess.run([sys.executable, 'import_vector_index.py', str(archive_path)], cwd=REPO, check=True,
               env={**os.environ, 'QDRANT_PATH': 'index/qdrant', 'QDRANT_URL': '', 'QDRANT_API_KEY': ''})
print('Existing indexes restored; no re-embedding.')

## Start Qdrant with the existing vectors

In [ ]:
import requests, tarfile, time, urllib.request
RUNTIME = REPO / 'qdrant-runtime'
RUNTIME.mkdir(exist_ok=True)
BINARY_DIR = Path('/tmp/legal-rag-qdrant-bin')
BINARY_DIR.mkdir(parents=True, exist_ok=True)
binary = BINARY_DIR / 'qdrant'
if not binary.exists():
    package = RUNTIME / 'qdrant.tar.gz'
    urllib.request.urlretrieve('https://github.com/qdrant/qdrant/releases/download/v1.19.0/qdrant-x86_64-unknown-linux-gnu.tar.gz', package)
    assert hashlib.sha256(package.read_bytes()).hexdigest() == 'e4405091f67d02f96fb941695ef8a6974e677632507ff7b04a3fcbb332ad9c19'
    with tarfile.open(package) as archive:
        member = next(m for m in archive.getmembers() if m.isfile() and Path(m.name).name == 'qdrant')
        with archive.extractfile(member) as source, binary.open('wb') as target:
            shutil.copyfileobj(source, target)
binary.chmod(0o755)
config = RUNTIME / 'config.yaml'
config.write_text('storage:\n  storage_path: ' + str(RUNTIME / 'storage') + '\nservice:\n  host: 127.0.0.1\n  http_port: 6333\n  grpc_port: 6334\n', encoding='utf-8')
try:
    running = requests.get('http://localhost:6333', timeout=2).json()
except requests.RequestException:
    running = None
if running is None:
    server_log = (RUNTIME / 'server.log').open('ab')
    server = subprocess.Popen([str(binary), '--config-path', str(config)], cwd=RUNTIME,
                              stdout=server_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if server.poll() is not None:
            raise RuntimeError((RUNTIME / 'server.log').read_text()[-2000:])
        try:
            running = requests.get('http://localhost:6333', timeout=2).json()
            break
        except requests.RequestException:
            time.sleep(1)
assert running and running['version'] == '1.19.0', 'Qdrant server did not start with the expected version.'
os.environ['QDRANT_PATH'] = ''
os.environ['QDRANT_URL'] = 'http://localhost:6333'
os.environ['QDRANT_API_KEY'] = ''
subprocess.run([sys.executable, '-u', 'upload_vector_index.py'], check=True)
print('Qdrant server ready with the existing 61,068 article vectors.')

## Optionally restore ablation checkpoints
Attach `ablation_top5_results.zip` through Add Input to resume. Kaggle may automatically unpack it; both layouts are supported. Existing working checkpoints take precedence.

In [ ]:
OUTPUT = REPO / 'results/ablation-top5'
checkpoint_zip = next(Path('/kaggle/input').rglob('ablation_top5_results.zip'), None)
if not OUTPUT.exists():
    if checkpoint_zip:
        with zipfile.ZipFile(checkpoint_zip) as archive:
            for name in archive.namelist():
                path = Path(name)
                assert name.startswith('ablation-top5/') and not path.is_absolute() and '..' not in path.parts and '\\' not in name and ':' not in name
            archive.extractall(REPO / 'results')
    else:
        checkpoint = next((p.parent for p in Path('/kaggle/input').rglob('records.jsonl')
                           if (p.parent / 'manifest.json').is_file()), None)
        if checkpoint:
            saved = json.loads((checkpoint / 'manifest.json').read_text())
            assert saved['protocol'] == 'fixed-candidates-top5-ablation-v1'
            OUTPUT.mkdir(parents=True)
            for name in ('manifest.json','records.jsonl','summary.json','comparison.md'):
                if (checkpoint / name).is_file():
                    shutil.copyfile(checkpoint / name, OUTPUT / name)
print('Ablation output:', OUTPUT)

## Run three ablations
Default: all 3,196 saved questions at top-5. Set `--limit` to a smaller number for a smoke run and choose a separate output folder. Every completed method record is checkpointed. Reports refresh every 25 questions and at completion.

In [ ]:
subprocess.run([sys.executable, '-u', 'benchmark_ablation.py', '--baseline', 'results/ablation-baseline',
                '--methods', 'bm25', 'dense', 'hybrid', '--require-gpu', '--limit', '0',
                '--output', 'results/ablation-top5'], cwd=REPO, check=True)
from IPython.display import Markdown, display
display(Markdown((OUTPUT / 'comparison.md').read_text()))

## Export full or partial checkpoints
Run after completion or after stopping inference. Partial raw records remain resumable even if the last report has not refreshed. Download the ZIP before discarding the runtime.

In [ ]:
EXPORT = Path('/kaggle/working/ablation_top5_results.zip')
assert OUTPUT.is_dir(), 'No ablation checkpoint directory yet.'
with zipfile.ZipFile(EXPORT, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT.rglob('*'):
        if path.is_file():
            archive.write(path, 'ablation-top5/' + path.relative_to(OUTPUT).as_posix())
print('Download:', EXPORT)